# Лабораторная работа 5. Переобучение, смещение–разброс и скользящий контроль

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 4 |
| Опора на лекции | лекция 4: переобучение (опр. 4.1), разложение смещение–разброс (теорема 4.3), отложенная выборка и $q$-кратный скользящий контроль (опр. 4.6–4.7), LOO, VC-размерность (опр. 4.10), принцип структурной минимизации риска (опр. 4.14); лекции 1–3: ERM, регуляризация, SVM |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 6 ч самостоятельно |

## Цель работы

Измерить смещение и разброс численно и убедиться, что теорема 4.3 выполняется с точностью до статистической погрешности; сравнить схемы скользящего контроля не по «удобству», а по смещению и дисперсии самой оценки; воспроизвести типовые утечки при валидации и измерить, насколько они завышают качество; проверить VC-размерность линейного классификатора перебором разметок и сравнить три стратегии выбора модели.

## Что нужно сдать

Заполненный ноутбук `lab05_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

Это методологическая работа курса. Всё, что делается дальше (работы 6–9), опирается
на протокол оценки качества, который выстраивается здесь. Ошибка в протоколе
обесценивает любые дальнейшие результаты: модель, «показавшая» AUC 0.95 из-за
утечки, в продакшене покажет 0.6.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from sklearn.model_selection import KFold, StratifiedKFold, ShuffleSplit, train_test_split
from labdata import load_personal

# Полиномы высоких степеней дают плохо обусловленную матрицу Вандермонда
# (работа 2, часть 4.2): scipy предупреждает об этом на каждом обучении.
# Предупреждение здесь ожидаемо и содержательно, но забивает вывод -- гасим его.
warnings.filterwarnings("ignore", message=".*Ill-conditioned matrix.*")

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=5)
describe_variant(variant)

---
# Часть 1. Кривые сложности и кривые обучения

Определение 4.1: переобучение — ситуация, когда $Q(a, X^\ell)$ мал, а $R(a)$ велик.
Два стандартных инструмента диагностики:

* **кривая сложности** (validation curve): ошибка на обучении и на контроле как
  функция параметра сложности;
* **кривая обучения** (learning curve): та же пара ошибок как функция размера
  обучающей выборки $\ell$.

Ось сложности задаётся вашим вариантом: `variant["complexity_axis"]`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# истинная зависимость и шум -- одни и те же на всю работу
F_TRUE = lambda x: np.sin(3 * x) + 0.4 * x ** 2
SIGMA = 0.35


def make_sample(n, generator):
    x = generator.uniform(-2, 2, n)
    return x[:, None], F_TRUE(x) + generator.normal(0, SIGMA, n)


AXIS = variant["complexity_axis"]
print("ось сложности вашего варианта:", AXIS)

if "полином" in AXIS:
    GRID = [1, 2, 3, 4, 5, 7, 9, 12, 15, 18]
    make_model = lambda p: make_pipeline(PolynomialFeatures(int(p)), Ridge(alpha=1e-8))
    XLABEL, LOGX = "степень полинома", False
elif "дерев" in AXIS:
    GRID = [1, 2, 3, 4, 5, 6, 8, 10, 14, 20]
    make_model = lambda p: DecisionTreeRegressor(max_depth=int(p), random_state=RANDOM_STATE)
    XLABEL, LOGX = "глубина дерева", False
elif "соседей" in AXIS:
    GRID = [60, 40, 25, 15, 10, 7, 5, 3, 2, 1]        # сложность растёт при убывании k
    make_model = lambda p: KNeighborsRegressor(n_neighbors=int(p))
    XLABEL, LOGX = "число соседей $k$ (сложность растёт справа налево)", False
else:
    GRID = list(np.logspace(-2, 4, 10))
    make_model = lambda p: make_pipeline(StandardScaler(), SVR(C=float(p), gamma="scale"))
    XLABEL, LOGX = "параметр $C$", True

print("сетка значений:", np.round(GRID, 4))

In [ ]:
gen = np.random.default_rng(RANDOM_STATE)
X_tr, y_tr = make_sample(60, gen)
X_te, y_te = make_sample(3000, gen)

# TODO: 1) кривая сложности: для каждого значения из GRID обучите make_model(p)
#          на (X_tr, y_tr) и посчитайте Q на обучении и на контроле;
#       2) кривая обучения: зафиксируйте достаточно сложную модель и для
#          размеров выборки [15, 20, 30, 45, 70, 110, 180, 300, 500]
#          усредните обе ошибки по 20 повторениям;
#       3) на обоих графиках проведите горизонталь на уровне sigma^2.

> **Вывод.** К какому уровню сходятся обе кривые обучения при $\ell \to \infty$ и почему именно к нему? Что означает большой зазор между кривыми и что — их схождение на высоком уровне?
>
> *(ваш ответ здесь)*

---
# Часть 2. Численная проверка теоремы 4.3

Теорема 4.3 (смещение–разброс):

$$
\mathbb{E}_{X^\ell,\varepsilon}\bigl[(y - a(x))^2\bigr]
= \underbrace{\bigl(f(x) - \mathbb{E}_{X^\ell}[a(x)]\bigr)^2}_{\text{смещение}^2}
+ \underbrace{\mathrm{Var}_{X^\ell}\bigl(a(x)\bigr)}_{\text{разброс}}
+ \underbrace{\sigma^2}_{\text{шум}} .
$$

Все три слагаемых **вычислимы**, если мы сами породили данные: $f$ и $\sigma$
известны, а математическое ожидание по $X^\ell$ оценивается многократной
генерацией обучающих выборок (`variant["n_repeats"]` повторений).

План: для каждого значения сложности из `GRID` обучить модель на $N$ независимых
выборках, усреднить предсказания по сетке точек $x$ и разложить ошибку на три части.

In [ ]:
N_REP = variant["n_repeats"]
N_TRAIN = 60
x_eval = np.linspace(-2, 2, 200)
f_eval = F_TRUE(x_eval)

# TODO: для каждого p из GRID:
#   1) N_REP раз сгенерируйте обучающую выборку размера N_TRAIN, обучите make_model(p)
#      и запомните предсказания на сетке x_eval -> матрица preds (N_REP x 200);
#   2) смещение^2 = среднее по сетке от (f_eval - preds.mean(axis=0))^2;
#      разброс     = среднее по сетке от preds.var(axis=0);
#      шум         = SIGMA^2;
#   3) отдельно ИЗМЕРЬТЕ E[(y - a(x))^2], добавив к f_eval гауссовский шум,
#      и сравните с суммой трёх слагаемых -- расхождение должно быть в пределах
#      статистической погрешности;
#   4) постройте график всех трёх вкладов и суммы в логарифмическом масштабе.

### Задание 2.2. Как выглядят сами предсказания

Для трёх значений сложности (слишком простая модель, оптимальная, слишком сложная)
нарисуйте: истинную зависимость $f$, среднее предсказание $\bar a$ (по всем
повторениям) и «облако» из 30 отдельных предсказаний. Смещение видно как
расхождение между $f$ и $\bar a$, разброс — как ширина облака.

In [ ]:
# TODO: для трёх значений сложности (простая / оптимальная / сложная) нарисуйте
#       30 отдельных предсказаний бледными линиями, среднее предсказание -- жирной,
#       истинную зависимость -- пунктиром. В заголовке укажите смещение^2 и разброс.

> **Вывод.** Совпала ли сумма трёх слагаемых с измеренной ошибкой? Как ведут себя смещение и разброс при росте сложности и где находится оптимум? Опишите словами, что видно на «облаке» предсказаний в каждом из трёх случаев.
>
> *(ваш ответ здесь)*

---
# Часть 3. Скользящий контроль

Определение 4.7: выборка делится на $q$ блоков, каждый по очереди служит контролем,

$$
\mathrm{CV}(\mu, X^L) = \frac1q\sum_{n=1}^{q} Q\bigl(a_n, X_n\bigr),
\qquad a_n = \mu(X^L \setminus X_n).
$$

Начнём с точного воспроизведения примера 4.9 из лекции: $y = (2,4,6,8)$,
модель — константа, метод А — выборочное среднее, метод Б — всегда $0$.
Лекция даёт $\mathrm{LOO}(A) \approx 8.89$ и $\mathrm{LOO}(Б) = 30$.

In [ ]:
y_toy = np.array([2.0, 4.0, 6.0, 8.0])

def loo(method, y):
    """Контроль по отдельным объектам: L обучений на L-1 объекте."""
    raise NotImplementedError


# TODO: воспроизведите таблицу примера 4.9 из лекции и оба значения LOO.

### Задание 3.2. Своя реализация $q$-кратного контроля

Реализуйте `my_kfold(n, q, shuffle, seed)`, возвращающий список пар индексов
`(train_idx, test_idx)`, и сверьте разбиения со `sklearn.model_selection.KFold`.
Затем реализуйте `cross_val_score` поверх своего разбиения.

In [ ]:
def my_kfold(n, q, shuffle=True, seed=0):
    """q непересекающихся блоков; возвращает пары (train_idx, test_idx).

    Подсказка: np.array_split делит массив индексов на почти равные части.
    """
    raise NotImplementedError


def my_cross_val_score(make_estimator, X, y, q=5, seed=0):
    raise NotImplementedError


# TODO: 1) сверьте разбиение со sklearn.model_selection.KFold при shuffle=False;
#       2) сравните среднее своей оценки со sklearn.model_selection.cross_val_score.

### Задание 3.3. Какая схема контроля лучше?

Схему контроля выбирают не по вкусу, а по свойствам самой **оценки**: она сама
случайна, и у неё есть смещение (относительно истинного риска) и дисперсия.

Проведите эксперимент: 200 раз сгенерируйте выборку из $L = 80$ объектов,
для каждой посчитайте оценку риска четырьмя схемами — отложенная выборка (25 %),
5-кратный, 10-кратный контроль и LOO — а также **истинный** риск на большой
независимой выборке. Сравните распределения ошибок оценивания.

In [ ]:
from sklearn.model_selection import LeaveOneOut

L, N_EXP = 80, 200
p_cv = GRID[len(GRID) // 2]
X_big, y_big = make_sample(20_000, np.random.default_rng(4242))

# TODO: N_EXP раз:
#   1) сгенерируйте выборку из L объектов;
#   2) посчитайте ИСТИННЫЙ риск обученной на ней модели (на большой выборке X_big);
#   3) посчитайте четыре оценки: отложенная 25%, KFold(5), KFold(10), LOO;
# TODO: сведите в таблицу смещение и стандартное отклонение каждой оценки,
#       постройте boxplot ошибок оценивания и диаграмму «оценка против истины».

> **Вывод.** У каких схем смещение значимо отличается от нуля и в какую сторону? Сравните разброс ошибки оценивания и стоимость. Почему на практике берут $q = 5$–$10$, а не LOO?
>
> *(ваш ответ здесь)*

---
# Часть 4. Утечки при валидации

Скользящий контроль даёт честную оценку **только если** ни один объект контрольного
блока не повлиял на обучение — включая предобработку. Нарушение этого правила
называется утечкой (data leakage) и завышает оценку, иногда драматически.

Сценарий вашего варианта: `variant["leak_scenario"]`.

Самая наглядная демонстрация — на **чистом шуме**: данные, в которых заведомо
нет никакой связи с целью. Честная оценка обязана дать AUC $\approx 0.5$.
Всё, что выше, — артефакт протокола.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

scenario = variant["leak_scenario"]
print("сценарий вашего варианта:", scenario)

g = np.random.default_rng(RANDOM_STATE)
n_obj, n_feat = 120, 4000
X_noise = g.normal(size=(n_obj, n_feat))
y_noise = g.integers(0, 2, n_obj)          # метки НЕ связаны с признаками

# TODO: посчитайте две оценки AUC по 5-кратному стратифицированному контролю:
#   (а) отбор 20 лучших признаков SelectKBest ПО ВСЕЙ выборке, затем CV;
#   (б) отбор признаков внутри Pipeline, то есть заново на каждом блоке.
# Истинное значение AUC для этих данных равно 0.5.

### Задание 4.2. Сценарий вашего варианта и величина завышения

Повторите ту же схему для сценария из вашего варианта на **осмысленных** данных
(своя выборка из работы 1): сравните честную оценку и оценку с утечкой,
измерьте разницу.

* *масштабирование до разбиения* — `StandardScaler` обучен на всей выборке;
* *отбор признаков по всей выборке* — как выше, но на реальных данных;
* *заполнение пропусков средним по всей выборке* — `SimpleImputer` обучен на всём.

In [ ]:
data = load_personal(variant, return_frame=True)
Xp, yp = np.vstack([data["X_train"], data["X_test"]]), np.r_[data["y_train"], data["y_test"]]
if data["task"] == "regression":
    yp = (yp > np.median(yp)).astype(int)

# TODO: 1) добавьте к признакам 500 чисто шумовых столбцов;
#       2) реализуйте свой сценарий утечки из variant["leak_scenario"] и честный
#          вариант того же протокола (всё внутри Pipeline);
#       3) сравните обе оценки AUC и посчитайте величину завышения.

> **Вывод.** Какое AUC дала утечка на чистом шуме и почему именно так получилось? Какой из трёх сценариев самый опасный и почему масштабирование «протекает» слабее, чем отбор признаков?
>
> *(ваш ответ здесь)*

---
# Часть 5. Вложенный скользящий контроль

Если по скользящему контролю **выбран** гиперпараметр, то само значение
$\min_\lambda \mathrm{CV}(\mu_\lambda)$ — уже не честная оценка риска: мы взяли
минимум по многим случайным величинам, а минимум смещён вниз. Это ровно та же
логика, что и в работе 2, часть 2: алгоритм, выбранный по выборке, показывает на
ней заниженную ошибку.

Лечение — **вложенный** контроль: внешний цикл оценивает качество, внутренний
подбирает гиперпараметр, и внутренний цикл не видит внешнего контрольного блока.

In [ ]:
from sklearn.model_selection import GridSearchCV

N_TRIALS, n_o, n_f = 30, 80, 100
grid = {"C": np.logspace(-4, 2, 10)}

# TODO: N_TRIALS раз сгенерируйте ЧИСТЫЙ ШУМ (X случайный, y случайный:
#       истинное AUC заведомо равно 0.5) и посчитайте две оценки:
#   1) наивную: GridSearchCV по сетке C с внешним StratifiedKFold(5), best_score_;
#   2) вложенную: внешний StratifiedKFold(5) для оценки, а внутри каждого блока --
#      свой GridSearchCV со StratifiedKFold(4) для подбора C.
# TODO: сравните средние обеих оценок с истиной 0.5 (со стандартной ошибкой),
#       затем посчитайте ПАРНУЮ разность (наивно - вложенно) с её стандартной
#       ошибкой и парным t-критерием scipy.stats.ttest_rel. Объясните в выводе,
#       почему парная разность разрешается статистически, а абсолютные
#       значения -- нет.
# TODO: постройте CV-кривую по сетке и boxplot обеих оценок.

> **Вывод.** На данных без всякой связи истинное AUC равно 0.5. Что показали обе оценки? От чего зависит величина смещения наивной схемы (длина сетки, размер выборки, сила сигнала)?
>
> *(ваш ответ здесь)*

---
# Часть 6. VC-размерность: проверка перебором

Определение 4.10: $\mathrm{VCdim}(A)$ — наибольшее $d$, для которого существует
набор из $d$ точек, разбиваемый моделью $A$ (то есть реализуются **все** $2^d$
разметок).

Пример 4.11: для линейных классификаторов в $\mathbb{R}^2$ VC-размерность равна 3.

Проверим это перебором. Линейная разделимость набора $(X, y)$ — это разрешимость
системы $y_i(w^{\mathsf T}x_i + b) \ge 1$, то есть **задача линейного
программирования** (нулевая целевая функция, важна лишь совместность).

In [ ]:
from itertools import product


def linearly_separable(X, y):
    """Существуют ли w, b с y_i (w^T x_i + b) >= 1?

    Подсказка: это задача ЛП с нулевой целевой функцией. Ограничения в форме
    A_ub @ z <= b_ub, где z = (w, b).
    """
    raise NotImplementedError


def shatters(X):
    """Реализуются ли все 2^d разметок набора точек X?"""
    raise NotImplementedError


# TODO: 1) проверьте, что 3 точки общего положения разбиваются, а 4 -- нет
#          (и для квадрата, и для случая «точка внутри треугольника»);
#          выведите конкретную нереализуемую разметку;
#       2) для n = 2, 3, 5 и d = 2..8 посчитайте долю реализуемых разметок
#          (усредните по 15 случайным наборам точек), сведите в таблицу
#          и постройте график с отметками d = n + 1.

> **Вывод.** Какая разметка четырёх точек оказалась нереализуемой в каждом из двух случаев? Где на графике доли разделимых разметок «ломается» кривая и как это связано с $\mathrm{VCdim} = n + 1$?
>
> *(ваш ответ здесь)*

---
# Часть 7. Три стратегии выбора модели

Определение 4.14 (SRM): для вложенных моделей $A_1 \subset \dots \subset A_M$
выбирается

$$
m^* = \arg\min_m \Bigl(Q(a_m, X^\ell) + \mathrm{penalty}(\mathrm{VCdim}(A_m), \ell)\Bigr).
$$

Сравните три стратегии выбора степени полинома:

1. по обучающей ошибке $Q(a_m, X^\ell)$;
2. по скользящему контролю $\mathrm{CV}(\mu_m)$;
3. по штрафу за сложность — в качестве практического аналога VC-штрафа возьмите
   информационные критерии
   $\mathrm{AIC} = \ell\ln\widehat{\sigma}^2 + 2p$ и
   $\mathrm{BIC} = \ell\ln\widehat{\sigma}^2 + p\ln\ell$, где $p$ — число параметров.

Эталон для сравнения — истинный риск на большой независимой выборке.

In [ ]:
gen = np.random.default_rng(RANDOM_STATE)
X_s, y_s = make_sample(50, gen)
X_ref, y_ref = make_sample(20_000, np.random.default_rng(999))
degrees = list(range(1, 16))

# TODO: для каждой степени посчитайте: Q на обучении, CV(5), AIC, BIC и
#       ИСТИННЫЙ риск на большой выборке. Выведите, какую степень выбирает
#       каждый критерий, и постройте график.

> **Вывод.** Какую степень выбрала каждая стратегия и какая ближе всего к оптимуму по истинному риску? Почему выбор по обучающей ошибке бесполезен, и чем BIC отличается от AIC?
>
> *(ваш ответ здесь)*

---
# Часть 8. Правильный протокол на своей выборке

Соберите всё вместе: сравните модели из работ 2–4 на индивидуальной выборке по
схеме контроля из вашего варианта (`variant["cv_scheme"]`), с полной
предобработкой **внутри** `Pipeline` и честной финальной оценкой на отложенной
контрольной выборке.

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

data = load_personal(variant)
# TODO: 1) при необходимости бинаризуйте цель по медиане ОБУЧАЮЩЕЙ выборки;
#       2) соберите схему контроля из variant["cv_scheme"];
#       3) для трёх моделей (логистическая регрессия, SVM с RBF, дерево решений)
#          подберите гиперпараметры GridSearchCV по этой схеме;
#       4) выведите таблицу: лучшие параметры, лучшее CV-значение и AUC на
#          отложенной контрольной выборке. Объясните разницу между двумя числами.

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Модель A: смещение большое, разброс малый. Модель B: наоборот. Какая из них выиграет при $\ell = 50$ и какая при $\ell = 50\,000$? Почему?
2. Почему оценка по отложенной выборке систематически пессимистична, а «лучшее значение по сетке гиперпараметров» — систематически оптимистично? Это два разных смещения или одно и то же?
3. Ваш коллега отобрал 50 признаков из 10 000 по корреляции с целью на всех данных, затем честно провёл 10-кратный контроль и получил AUC 0.85. Что вы ему скажете и как перепроверите результат?
4. VC-размерность линейного классификатора в $\mathbb{R}^n$ равна $n+1$. Значит ли это, что VC-размерность всегда равна числу параметров? Приведите контрпример или объясните, почему это так.
5. Вы получили CV AUC = 0.91 и AUC на отложенной выборке = 0.86. Это нормально или признак ошибки? Какие три причины такого разрыва вы проверите в первую очередь?

### Домашнее задание

1. **$5\times2$-кратный контроль и статистическая значимость.** Реализуйте схему $5\times2$cv (пять независимых разбиений пополам) и постройте на её основе парный $t$-тест для сравнения двух моделей (Dietterich, 1998). Сравните с наивным подходом «10-кратный контроль + парный $t$-тест по блокам» на данных, где обе модели заведомо **одинаковы** (например, две копии одной модели с разным `random_state`): покажите, что наивный тест отвергает нулевую гипотезу существенно чаще заявленных 5 %, и объясните почему (подсказка: обучающие выборки в блоках перекрываются, поэтому ошибки зависимы).

2. **Кривая обучения и стоимость данных.** Для своей выборки постройте кривую обучения для лучшей модели из части 8, аппроксимируйте её степенным законом $R(\ell) \approx R_\infty + c\,\ell^{-\alpha}$ (подгонка по МНК в логарифмических координатах) и оцените, сколько объектов нужно, чтобы уменьшить ошибку ещё на 10 %. Обсудите, когда выгоднее собирать данные, а когда — усложнять модель, опираясь на разложение из части 2.